# Differential Geometry of Curves and Surfaces
## Curvature, Torsion, Fundamental Forms, and Geodesics — With Applications to Trajectory Smoothing

This notebook provides a complete, from-scratch treatment of **differential geometry** — the mathematical language for studying curves and surfaces. We implement curvature, torsion, fundamental forms, geodesics, and apply them to trajectory smoothing on curved surfaces.

**What you'll learn:**
1. Parametric curves: arc length, curvature, torsion, and the Frenet-Serret frame
2. Surfaces: first and second fundamental forms, Gaussian and mean curvature
3. Geodesics on surfaces via variational methods and Christoffel symbols
4. Application: trajectory smoothing on curved surfaces for robotics

**Prerequisites:** Multivariable calculus (gradients, partial derivatives), linear algebra, calculus of variations (Euler-Lagrange equations).

**References:**
- do Carmo, M.P. *Differential Geometry of Curves and Surfaces*, Dover, 2016.
- Pressley, A. *Elementary Differential Geometry*, 2nd ed., Springer, 2010.
- See also: [Lie Groups notebook](../../maths/linear-algebra/lie-groups/lie_groups.ipynb), [Calculus of Variations notebook](../../maths/calculus-of-variations/calculus_of_variations.ipynb).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.integrate import solve_ivp
from scipy.interpolate import CubicSpline
from scipy.optimize import minimize

%matplotlib inline

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2,
})

np.random.seed(42)

In [ ]:
# =============================================================================
# Global Constants
# =============================================================================

N_CURVE_POINTS = 500      # Default points for curve discretization
HELIX_RADIUS = 1.0        # Helix radius
HELIX_PITCH = 0.3         # Helix pitch (height per revolution / 2pi)

N_SURFACE_GRID = 80       # Grid resolution for surface plots
FINITE_DIFF_EPS = 1e-6    # Step size for finite differences

GEODESIC_MAX_S = 10.0     # Max arc length for geodesic shooting
N_GEODESIC_POINTS = 1000  # Points for geodesic integration

TOL = 1e-6

# Plot colors
C_BLUE = 'steelblue'
C_RED = 'coral'
C_GREEN = 'seagreen'
C_GOLD = 'goldenrod'
C_PURPLE = 'mediumpurple'

---
## 1. Introduction: Why Differential Geometry?

Robotics lives on **curved spaces**. A robot arm's configuration space is a torus $T^n$, not $\mathbb{R}^n$. A quadrotor's attitude lives on $SO(3)$. Even in Euclidean workspace, trajectories are **curves** and obstacle boundaries are **surfaces** — objects whose intrinsic properties (curvature, geodesics) govern motion planning and control.

| Concept | Euclidean Approach | Differential Geometry |
|---------|-------------------|----------------------|
| Straight line | Shortest path in $\mathbb{R}^n$ | **Geodesic** on a manifold |
| Distance | $\|x - y\|_2$ | Arc length via the **metric tensor** |
| Turning rate | Discrete heading changes | **Curvature** $\kappa(s)$ |
| Surface area | Sum of triangle areas | Integral of $\sqrt{EG - F^2}$ |
| Shortest path on surface | Discretize + Dijkstra | Solve the **geodesic equation** |

**Roadmap:** We start with curves in $\mathbb{R}^3$ (arc length, curvature, torsion, Frenet frame), move to surfaces (fundamental forms, Gaussian curvature), study geodesics, and finish with a trajectory smoothing application.

---
## 2. Parametric Curves and Arc Length

A **parametric curve** in $\mathbb{R}^3$ is a smooth map $\alpha: I \to \mathbb{R}^3$ where $I \subset \mathbb{R}$ is an interval. A curve is **regular** if $\alpha'(t) \neq 0$ for all $t \in I$.

### Arc Length

The **arc length** from $t = a$ to $t = b$ is:

$$s(t) = \int_a^t \|\alpha'(\tau)\| \, d\tau$$

This measures the "distance traveled" along the curve, independent of parametrization speed.

### Arc Length Parametrization

A curve is **unit-speed** (parametrized by arc length) if $\|\alpha'(s)\| = 1$ for all $s$. Every regular curve admits an arc-length reparametrization: define $s(t)$ as above, invert to get $t(s)$, and set $\beta(s) = \alpha(t(s))$.

**Why it matters in robotics:** Arc-length parametrization decouples the *geometry* of a path (its shape) from the *timing* (how fast the robot traverses it). Trajectory planning often first designs a geometric path, then assigns a time profile.

In [ ]:
def compute_arc_length(points):
    """Compute cumulative arc length along a discrete curve.

    Uses the trapezoidal approximation: sum of segment lengths.

    Args:
        points: Curve sample points. Shape: (N, d) where d is dimension.

    Returns:
        s: Cumulative arc length at each point. Shape: (N,). s[0] = 0.
    """
    diffs = np.diff(points, axis=0)                    # (N-1, d)
    segment_lengths = np.linalg.norm(diffs, axis=1)    # (N-1,)
    s = np.zeros(len(points))
    s[1:] = np.cumsum(segment_lengths)
    return s


def arc_length_reparametrize(curve_func, t_array, n_out=None):
    """Reparametrize a curve by arc length via interpolation.

    Args:
        curve_func: Callable t -> point in R^d. Takes scalar, returns shape (d,).
        t_array: Parameter values to sample. Shape: (N,).
        n_out: Number of output points (default: same as input).

    Returns:
        s_uniform: Uniform arc-length parameter values. Shape: (n_out,).
        points_out: Reparametrized curve points. Shape: (n_out, d).
    """
    if n_out is None:
        n_out = len(t_array)

    # Sample the curve
    points = np.array([curve_func(t) for t in t_array])
    s = compute_arc_length(points)

    # Build interpolant: t(s)
    t_of_s = CubicSpline(s, t_array)

    # Uniform arc-length samples
    s_uniform = np.linspace(0, s[-1], n_out)
    t_uniform = t_of_s(s_uniform)
    points_out = np.array([curve_func(t) for t in t_uniform])

    return s_uniform, points_out


# ---- Define example curves ----

def helix(t):
    """Circular helix: (a*cos(t), a*sin(t), b*t)."""
    return np.array([HELIX_RADIUS * np.cos(t),
                     HELIX_RADIUS * np.sin(t),
                     HELIX_PITCH * t])

def lissajous(t):
    """Lissajous figure in 3D."""
    return np.array([np.sin(2 * t), np.sin(3 * t), 0.3 * np.sin(5 * t)])

def trefoil_knot(t):
    """Trefoil knot parametrization."""
    return np.array([np.sin(t) + 2 * np.sin(2 * t),
                     np.cos(t) - 2 * np.cos(2 * t),
                     -np.sin(3 * t)])

In [ ]:
# ---- Visualize curves colored by arc length ----
curves = [
    ("Circular Helix", helix, np.linspace(0, 4 * np.pi, N_CURVE_POINTS)),
    ("Lissajous Figure", lissajous, np.linspace(0, 2 * np.pi, N_CURVE_POINTS)),
    ("Trefoil Knot", trefoil_knot, np.linspace(0, 2 * np.pi, N_CURVE_POINTS)),
]

fig = plt.figure(figsize=(18, 5))
for idx, (name, curve_func, t_arr) in enumerate(curves):
    points = np.array([curve_func(t) for t in t_arr])
    s = compute_arc_length(points)

    ax = fig.add_subplot(1, 3, idx + 1, projection='3d')
    sc = ax.scatter(points[:, 0], points[:, 1], points[:, 2],
                    c=s, cmap='viridis', s=2)
    ax.set_title(name)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_zlabel('z')
    plt.colorbar(sc, ax=ax, label='Arc length s', shrink=0.6)

plt.suptitle('Parametric Curves Colored by Arc Length', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Verify arc-length reparametrization: after reparametrization, speed should be ~1
t_test = np.linspace(0, 4 * np.pi, N_CURVE_POINTS)
s_reparam, pts_reparam = arc_length_reparametrize(helix, t_test)
speeds = np.linalg.norm(np.diff(pts_reparam, axis=0), axis=1) / np.diff(s_reparam)
speed_error = np.max(np.abs(speeds - 1.0))
status = "PASS" if speed_error < 0.01 else "FAIL"
print(f"Arc-length reparametrization — max |speed - 1|: {speed_error:.6f} [{status}]")

---
## 3. Curvature and Torsion

For a curve $\alpha(t)$ in $\mathbb{R}^3$, the **curvature** measures how fast the curve turns and the **torsion** measures how fast it twists out of its osculating plane.

### Formulas for Arbitrary Parametrization

For a unit-speed curve ($\|\alpha'(s)\| = 1$), curvature is simply $\kappa(s) = \|\alpha''(s)\|$. For an **arbitrary** parametrization, we need the general formulas:

$$\kappa(t) = \frac{\|\alpha'(t) \times \alpha''(t)\|}{\|\alpha'(t)\|^3}$$

$$\tau(t) = \frac{(\alpha'(t) \times \alpha''(t)) \cdot \alpha'''(t)}{\|\alpha'(t) \times \alpha''(t)\|^2}$$

### Derivation

Starting from $\alpha(t) = \beta(s(t))$ where $\beta$ is the arc-length parametrization:
- $\alpha' = \dot{s} \, T$ where $\dot{s} = \|\alpha'\|$ and $T$ is the unit tangent
- $\alpha'' = \ddot{s} \, T + \dot{s}^2 \kappa \, N$ where $N$ is the unit normal
- $\alpha' \times \alpha'' = \dot{s}^3 \kappa \, B$ where $B = T \times N$ is the binormal

Taking norms: $\|\alpha' \times \alpha''\| = \dot{s}^3 \kappa$, giving the curvature formula.

For torsion, differentiating once more and using the Frenet-Serret equations yields the torsion formula.

### Teaching Example: Circular Helix

For the helix $\alpha(t) = (a\cos t, a\sin t, bt)$, the analytic values are:

$$\kappa = \frac{a}{a^2 + b^2}, \qquad \tau = \frac{b}{a^2 + b^2}$$

Both are **constant** — the helix has uniform curvature and torsion.

In [ ]:
def compute_curvature_torsion(points):
    """Compute curvature and torsion of a discrete 3D curve.

    Uses central finite differences for derivatives up to third order.

    Args:
        points: Curve sample points (equally spaced in parameter). Shape: (N, 3).

    Returns:
        kappa: Curvature at interior points. Shape: (N-4,).
        tau: Torsion at interior points. Shape: (N-4,).
        indices: Indices into original array for valid points. Shape: (N-4,).
    """
    N = len(points)
    # First derivative (central difference)
    d1 = np.gradient(points, axis=0)       # (N, 3)
    # Second derivative
    d2 = np.gradient(d1, axis=0)           # (N, 3)
    # Third derivative
    d3 = np.gradient(d2, axis=0)           # (N, 3)

    # Trim edges where finite differences are less accurate
    trim = 2
    d1 = d1[trim:-trim]
    d2 = d2[trim:-trim]
    d3 = d3[trim:-trim]
    indices = np.arange(trim, N - trim)

    # Cross product alpha' x alpha''
    cross12 = np.cross(d1, d2)             # (M, 3)
    cross12_norm = np.linalg.norm(cross12, axis=1)  # (M,)
    d1_norm = np.linalg.norm(d1, axis=1)            # (M,)

    # Curvature: |alpha' x alpha''| / |alpha'|^3
    kappa = cross12_norm / (d1_norm ** 3 + 1e-30)

    # Torsion: (alpha' x alpha'') . alpha''' / |alpha' x alpha''|^2
    tau = np.sum(cross12 * d3, axis=1) / (cross12_norm ** 2 + 1e-30)

    return kappa, tau, indices


# ---- Verify on circular helix ----
t_helix = np.linspace(0, 4 * np.pi, N_CURVE_POINTS)
pts_helix = np.array([helix(t) for t in t_helix])

kappa_helix, tau_helix, idx_helix = compute_curvature_torsion(pts_helix)

# Analytic values
a, b = HELIX_RADIUS, HELIX_PITCH
kappa_exact = a / (a**2 + b**2)
tau_exact = b / (a**2 + b**2)

kappa_mean = np.mean(kappa_helix[10:-10])  # trim further for accuracy
tau_mean = np.mean(tau_helix[10:-10])

status_k = "PASS" if abs(kappa_mean - kappa_exact) / kappa_exact < 0.01 else "FAIL"
status_t = "PASS" if abs(tau_mean - tau_exact) / tau_exact < 0.01 else "FAIL"
print(f"Helix curvature — analytic: {kappa_exact:.6f}, numerical: {kappa_mean:.6f} [{status_k}]")
print(f"Helix torsion   — analytic: {tau_exact:.6f}, numerical: {tau_mean:.6f} [{status_t}]")

In [ ]:
# ---- Curvature visualization ----
s_helix = compute_arc_length(pts_helix)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Left: helix colored by curvature
ax = axes[0]
ax = fig.add_subplot(1, 3, 1, projection='3d')
sc = ax.scatter(pts_helix[idx_helix, 0], pts_helix[idx_helix, 1], pts_helix[idx_helix, 2],
                c=kappa_helix, cmap='magma', s=3)
ax.set_title('Helix Colored by $\\kappa$')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
plt.colorbar(sc, ax=ax, label='$\\kappa$', shrink=0.6)

# Middle: curvature vs arc length
ax = axes[1]
ax.plot(s_helix[idx_helix], kappa_helix, color=C_BLUE, label='Numerical $\\kappa(s)$')
ax.axhline(kappa_exact, color=C_RED, linestyle='--', label=f'Analytic $\\kappa = {kappa_exact:.4f}$')
ax.set_xlabel('Arc length $s$')
ax.set_ylabel('Curvature $\\kappa$')
ax.set_title('Curvature vs Arc Length')
ax.legend()

# Right: torsion vs arc length
ax = axes[2]
ax.plot(s_helix[idx_helix], tau_helix, color=C_GREEN, label='Numerical $\\tau(s)$')
ax.axhline(tau_exact, color=C_RED, linestyle='--', label=f'Analytic $\\tau = {tau_exact:.4f}$')
ax.set_xlabel('Arc length $s$')
ax.set_ylabel('Torsion $\\tau$')
ax.set_title('Torsion vs Arc Length')
ax.legend()

plt.suptitle('Curvature and Torsion of a Circular Helix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. The Frenet-Serret Frame

At each point of a regular curve with $\kappa \neq 0$, there exists a natural orthonormal basis called the **Frenet-Serret frame** $\{T, N, B\}$:

| Vector | Name | Definition | Intuition |
|--------|------|------------|-----------|
| $T$ | Unit tangent | $T = \alpha'/\|\alpha'\|$ | Direction of travel |
| $N$ | Principal normal | $N = T'/\|T'\|$ | Direction the curve is turning |
| $B$ | Binormal | $B = T \times N$ | Normal to the osculating plane |

### The Frenet-Serret Equations

The frame evolves along the curve according to:

$$\frac{d}{ds}\begin{bmatrix} T \\ N \\ B \end{bmatrix} = \begin{bmatrix} 0 & \kappa & 0 \\ -\kappa & 0 & \tau \\ 0 & -\tau & 0 \end{bmatrix} \begin{bmatrix} T \\ N \\ B \end{bmatrix}$$

This is a **skew-symmetric** matrix — which makes sense because the frame is orthonormal and the matrix generates rotations in $SO(3)$. The curvature $\kappa$ drives rotation in the $T$-$N$ plane (turning), while the torsion $\tau$ drives rotation in the $N$-$B$ plane (twisting).

**Connection to Lie theory:** The Frenet-Serret equations are an ODE on $SO(3)$: $\dot{R} = R \, \Omega$ where $\Omega$ is the skew-symmetric matrix above. This is exactly the structure studied in the [Lie Groups notebook](../../maths/linear-algebra/lie-groups/lie_groups.ipynb).

In [ ]:
def frenet_serret_frame(points):
    """Compute the Frenet-Serret frame (T, N, B) at each point of a 3D curve.

    Args:
        points: Curve sample points (equally spaced). Shape: (N, 3).

    Returns:
        T: Unit tangent vectors. Shape: (M, 3).
        N: Unit normal vectors. Shape: (M, 3).
        B: Binormal vectors. Shape: (M, 3).
        indices: Valid indices into original array. Shape: (M,).
    """
    # First derivative
    d1 = np.gradient(points, axis=0)   # (N, 3)
    d1_norm = np.linalg.norm(d1, axis=1, keepdims=True)  # (N, 1)
    T_all = d1 / (d1_norm + 1e-30)    # (N, 3)

    # Derivative of T (for normal direction)
    dT = np.gradient(T_all, axis=0)    # (N, 3)
    dT_norm = np.linalg.norm(dT, axis=1, keepdims=True)
    N_all = dT / (dT_norm + 1e-30)

    # Binormal
    B_all = np.cross(T_all, N_all)
    B_norm = np.linalg.norm(B_all, axis=1, keepdims=True)
    B_all = B_all / (B_norm + 1e-30)

    # Re-orthogonalize N to ensure exact orthonormality
    N_all = np.cross(B_all, T_all)

    # Trim edges
    trim = 3
    indices = np.arange(trim, len(points) - trim)
    return T_all[indices], N_all[indices], B_all[indices], indices


# ---- Compute Frenet frame for helix ----
T_h, N_h, B_h, idx_fs = frenet_serret_frame(pts_helix)

# Verify orthonormality
dot_TN = np.abs(np.sum(T_h * N_h, axis=1))
dot_TB = np.abs(np.sum(T_h * B_h, axis=1))
dot_NB = np.abs(np.sum(N_h * B_h, axis=1))

max_dot = max(dot_TN.max(), dot_TB.max(), dot_NB.max())
status = "PASS" if max_dot < 1e-3 else "FAIL"
print(f"Orthonormality check — max |T·N|, |T·B|, |N·B|: {max_dot:.6e} [{status}]")

norm_T = np.linalg.norm(T_h, axis=1)
norm_N = np.linalg.norm(N_h, axis=1)
norm_B = np.linalg.norm(B_h, axis=1)
max_norm_err = max(np.max(np.abs(norm_T - 1)), np.max(np.abs(norm_N - 1)), np.max(np.abs(norm_B - 1)))
status2 = "PASS" if max_norm_err < 1e-3 else "FAIL"
print(f"Unit vector check   — max ||v| - 1|: {max_norm_err:.6e} [{status2}]")

In [ ]:
# ---- Visualize Frenet frame on helix ----
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

# Plot curve
ax.plot(pts_helix[:, 0], pts_helix[:, 1], pts_helix[:, 2], color='gray', alpha=0.5, linewidth=1)

# Draw frames at selected points
step = len(idx_fs) // 12
arrow_len = 0.35
for i in range(0, len(idx_fs), step):
    p = pts_helix[idx_fs[i]]
    ax.quiver(*p, *T_h[i] * arrow_len, color=C_BLUE, arrow_length_ratio=0.15, linewidth=2)
    ax.quiver(*p, *N_h[i] * arrow_len, color=C_RED, arrow_length_ratio=0.15, linewidth=2)
    ax.quiver(*p, *B_h[i] * arrow_len, color=C_GREEN, arrow_length_ratio=0.15, linewidth=2)

# Legend via dummy plots
ax.plot([], [], color=C_BLUE, linewidth=2, label='T (tangent)')
ax.plot([], [], color=C_RED, linewidth=2, label='N (normal)')
ax.plot([], [], color=C_GREEN, linewidth=2, label='B (binormal)')
ax.legend(fontsize=12)

ax.set_title('Frenet-Serret Frame on a Circular Helix', fontsize=14, fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
plt.tight_layout()
plt.show()

---
## 5. Surfaces and the First Fundamental Form

A **parametric surface** is a smooth map $\sigma: U \subset \mathbb{R}^2 \to \mathbb{R}^3$, written as $\sigma(u, v) = (x(u,v), y(u,v), z(u,v))$.

### The First Fundamental Form (Metric Tensor)

The **first fundamental form** encodes the intrinsic geometry — lengths, angles, and areas — on the surface. It is defined by the metric coefficients:

$$E = \sigma_u \cdot \sigma_u, \quad F = \sigma_u \cdot \sigma_v, \quad G = \sigma_v \cdot \sigma_v$$

where $\sigma_u = \partial\sigma/\partial u$ and $\sigma_v = \partial\sigma/\partial v$ are the tangent vectors.

### Arc Length on a Surface

For a curve $\gamma(t) = \sigma(u(t), v(t))$ on the surface:

$$L = \int_a^b \sqrt{E\dot{u}^2 + 2F\dot{u}\dot{v} + G\dot{v}^2} \, dt$$

### Area Element

$$dA = \sqrt{EG - F^2} \, du \, dv = \|\sigma_u \times \sigma_v\| \, du \, dv$$

The total area is $A = \iint_U \sqrt{EG - F^2} \, du \, dv$.

In [ ]:
def first_fundamental_form(sigma_func, u, v, eps=FINITE_DIFF_EPS):
    """Compute the first fundamental form coefficients E, F, G at a point.

    Uses central finite differences for partial derivatives.

    Args:
        sigma_func: Surface parametrization (u, v) -> R^3. Callable.
        u: First parameter value. Scalar.
        v: Second parameter value. Scalar.
        eps: Finite difference step size. Scalar.

    Returns:
        E: Coefficient E = sigma_u . sigma_u. Scalar.
        F: Coefficient F = sigma_u . sigma_v. Scalar.
        G: Coefficient G = sigma_v . sigma_v. Scalar.
    """
    # Partial derivatives via central differences
    sigma_u = (sigma_func(u + eps, v) - sigma_func(u - eps, v)) / (2 * eps)
    sigma_v = (sigma_func(u, v + eps) - sigma_func(u, v - eps)) / (2 * eps)

    E = np.dot(sigma_u, sigma_u)
    F = np.dot(sigma_u, sigma_v)
    G = np.dot(sigma_v, sigma_v)
    return E, F, G


def surface_area(sigma_func, u_range, v_range, N=100):
    """Compute the area of a parametric surface via numerical integration.

    Uses the midpoint rule on a uniform grid.

    Args:
        sigma_func: Surface parametrization (u, v) -> R^3.
        u_range: Tuple (u_min, u_max).
        v_range: Tuple (v_min, v_max).
        N: Grid resolution per dimension. Integer.

    Returns:
        area: Approximate surface area. Scalar.
    """
    u_vals = np.linspace(u_range[0], u_range[1], N)
    v_vals = np.linspace(v_range[0], v_range[1], N)
    du = u_vals[1] - u_vals[0]
    dv = v_vals[1] - v_vals[0]

    area = 0.0
    eps = FINITE_DIFF_EPS
    for u in u_vals:
        for v in v_vals:
            sigma_u = (sigma_func(u + eps, v) - sigma_func(u - eps, v)) / (2 * eps)
            sigma_v = (sigma_func(u, v + eps) - sigma_func(u, v - eps)) / (2 * eps)
            cross = np.cross(sigma_u, sigma_v)
            area += np.linalg.norm(cross) * du * dv
    return area


# ---- Define surface parametrizations ----

def sphere(u, v, r=1.0):
    """Sphere of radius r: u in [0, pi], v in [0, 2*pi]."""
    return np.array([r * np.sin(u) * np.cos(v),
                     r * np.sin(u) * np.sin(v),
                     r * np.cos(u)])

def torus(u, v, R=2.0, r=0.7):
    """Torus: u, v in [0, 2*pi]. R = major radius, r = minor radius."""
    return np.array([(R + r * np.cos(v)) * np.cos(u),
                     (R + r * np.cos(v)) * np.sin(u),
                     r * np.sin(v)])


# ---- Verify sphere area ----
r_sphere = 1.5
sphere_r = lambda u, v: sphere(u, v, r=r_sphere)
area_numerical = surface_area(sphere_r, (0.01, np.pi - 0.01), (0, 2 * np.pi), N=80)
area_exact = 4 * np.pi * r_sphere**2
rel_err = abs(area_numerical - area_exact) / area_exact
status = "PASS" if rel_err < 0.02 else "FAIL"
print(f"Sphere area (r={r_sphere}) — exact: {area_exact:.4f}, numerical: {area_numerical:.4f}, "
      f"rel error: {rel_err:.4%} [{status}]")

In [ ]:
# ---- Visualize sphere and torus ----
fig = plt.figure(figsize=(14, 6))

# Sphere
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
u_grid = np.linspace(0, np.pi, 50)
v_grid = np.linspace(0, 2 * np.pi, 50)
U, V = np.meshgrid(u_grid, v_grid)
X_s = np.sin(U) * np.cos(V)
Y_s = np.sin(U) * np.sin(V)
Z_s = np.cos(U)
ax1.plot_surface(X_s, Y_s, Z_s, cmap='coolwarm', alpha=0.7, edgecolor='none')
ax1.set_title('Unit Sphere\n$E=1, F=0, G=\\sin^2 u$')
ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.set_zlabel('z')

# Torus
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
R_t, r_t = 2.0, 0.7
u_grid = np.linspace(0, 2 * np.pi, 60)
v_grid = np.linspace(0, 2 * np.pi, 60)
U, V = np.meshgrid(u_grid, v_grid)
X_t = (R_t + r_t * np.cos(V)) * np.cos(U)
Y_t = (R_t + r_t * np.cos(V)) * np.sin(U)
Z_t = r_t * np.sin(V)
ax2.plot_surface(X_t, Y_t, Z_t, cmap='viridis', alpha=0.7, edgecolor='none')
ax2.set_title(f'Torus ($R={R_t}, r={r_t}$)\n$E=(R+r\\cos v)^2, F=0, G=r^2$')
ax2.set_xlabel('x'); ax2.set_ylabel('y'); ax2.set_zlabel('z')

plt.suptitle('Parametric Surfaces', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. Second Fundamental Form and Curvatures

The **second fundamental form** captures how the surface curves in the ambient space $\mathbb{R}^3$.

### Unit Normal

$$\hat{n} = \frac{\sigma_u \times \sigma_v}{\|\sigma_u \times \sigma_v\|}$$

### Second Fundamental Form Coefficients

$$L = \sigma_{uu} \cdot \hat{n}, \quad M = \sigma_{uv} \cdot \hat{n}, \quad N = \sigma_{vv} \cdot \hat{n}$$

### Shape Operator and Principal Curvatures

The **shape operator** (Weingarten map) has matrix representation:

$$S = \begin{bmatrix} E & F \\ F & G \end{bmatrix}^{-1} \begin{bmatrix} L & M \\ M & N \end{bmatrix}$$

The **principal curvatures** $\kappa_1, \kappa_2$ are the eigenvalues of $S$.

### Gaussian and Mean Curvature

$$K = \kappa_1 \kappa_2 = \frac{LN - M^2}{EG - F^2} \qquad \text{(Gaussian curvature)}$$

$$H = \frac{\kappa_1 + \kappa_2}{2} = \frac{EN - 2FM + GL}{2(EG - F^2)} \qquad \text{(Mean curvature)}$$

**Gauss's Theorema Egregium:** $K$ depends only on $E, F, G$ and their derivatives — it is an **intrinsic** quantity, independent of how the surface is embedded in $\mathbb{R}^3$. This is remarkable: a being living on the surface can measure $K$ without leaving the surface.

In [ ]:
def gaussian_mean_curvature(sigma_func, u, v, eps=FINITE_DIFF_EPS):
    """Compute Gaussian and mean curvature at a point on a parametric surface.

    Args:
        sigma_func: Surface parametrization (u, v) -> R^3. Callable.
        u: First parameter. Scalar.
        v: Second parameter. Scalar.
        eps: Finite difference step. Scalar.

    Returns:
        K: Gaussian curvature. Scalar.
        H: Mean curvature. Scalar.
    """
    # First derivatives
    sigma_u = (sigma_func(u + eps, v) - sigma_func(u - eps, v)) / (2 * eps)
    sigma_v = (sigma_func(u, v + eps) - sigma_func(u, v - eps)) / (2 * eps)

    # Second derivatives
    sigma_uu = (sigma_func(u + eps, v) - 2 * sigma_func(u, v) + sigma_func(u - eps, v)) / eps**2
    sigma_vv = (sigma_func(u, v + eps) - 2 * sigma_func(u, v) + sigma_func(u, v - eps)) / eps**2
    sigma_uv = (sigma_func(u + eps, v + eps) - sigma_func(u + eps, v - eps)
                - sigma_func(u - eps, v + eps) + sigma_func(u - eps, v - eps)) / (4 * eps**2)

    # Unit normal
    n_vec = np.cross(sigma_u, sigma_v)
    n_hat = n_vec / (np.linalg.norm(n_vec) + 1e-30)

    # First fundamental form
    E = np.dot(sigma_u, sigma_u)
    F = np.dot(sigma_u, sigma_v)
    G = np.dot(sigma_v, sigma_v)

    # Second fundamental form
    L = np.dot(sigma_uu, n_hat)
    M = np.dot(sigma_uv, n_hat)
    N_coeff = np.dot(sigma_vv, n_hat)

    # Gaussian and mean curvature
    denom = E * G - F**2
    K = (L * N_coeff - M**2) / (denom + 1e-30)
    H = (E * N_coeff - 2 * F * M + G * L) / (2 * denom + 1e-30)

    return K, H


# ---- Verify on unit sphere: K = 1/r^2, H = 1/r ----
r_test = 2.0
sphere_test = lambda u, v: sphere(u, v, r=r_test)

K_test, H_test = gaussian_mean_curvature(sphere_test, np.pi / 3, np.pi / 4)
K_exact = 1.0 / r_test**2
H_exact = 1.0 / r_test

status_K = "PASS" if abs(K_test - K_exact) / K_exact < 0.01 else "FAIL"
status_H = "PASS" if abs(H_test - H_exact) / H_exact < 0.01 else "FAIL"
print(f"Sphere (r={r_test}) Gaussian curvature — exact: {K_exact:.6f}, numerical: {K_test:.6f} [{status_K}]")
print(f"Sphere (r={r_test}) mean curvature     — exact: {H_exact:.6f}, numerical: {H_test:.6f} [{status_H}]")

In [ ]:
# ---- Gaussian curvature maps: torus and saddle ----
fig = plt.figure(figsize=(16, 7))

# Torus K map
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
N_grid = 60
u_g = np.linspace(0, 2 * np.pi, N_grid)
v_g = np.linspace(0, 2 * np.pi, N_grid)
Ug, Vg = np.meshgrid(u_g, v_g)

# Compute Gaussian curvature on torus grid
K_torus = np.zeros_like(Ug)
for i in range(N_grid):
    for j in range(N_grid):
        K_torus[i, j], _ = gaussian_mean_curvature(torus, Ug[i, j], Vg[i, j])

R_t, r_t = 2.0, 0.7
X_t = (R_t + r_t * np.cos(Vg)) * np.cos(Ug)
Y_t = (R_t + r_t * np.cos(Vg)) * np.sin(Ug)
Z_t = r_t * np.sin(Vg)

# Normalize K for colormap
K_norm = (K_torus - K_torus.min()) / (K_torus.max() - K_torus.min() + 1e-30)
colors_t = plt.cm.RdBu_r(K_norm)
ax1.plot_surface(X_t, Y_t, Z_t, facecolors=colors_t, alpha=0.9, edgecolor='none')
ax1.set_title('Torus: Gaussian Curvature $K$\n(red = $K>0$, blue = $K<0$)')
ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.set_zlabel('z')

# Saddle surface z = x^2 - y^2
ax2 = fig.add_subplot(1, 2, 2, projection='3d')

def saddle(u, v):
    """Saddle surface (hyperbolic paraboloid): z = u^2 - v^2."""
    return np.array([u, v, u**2 - v**2])

u_s = np.linspace(-1.5, 1.5, N_grid)
v_s = np.linspace(-1.5, 1.5, N_grid)
Us, Vs = np.meshgrid(u_s, v_s)

K_saddle = np.zeros_like(Us)
for i in range(N_grid):
    for j in range(N_grid):
        K_saddle[i, j], _ = gaussian_mean_curvature(saddle, Us[i, j], Vs[i, j])

Z_saddle = Us**2 - Vs**2
K_norm_s = (K_saddle - K_saddle.min()) / (K_saddle.max() - K_saddle.min() + 1e-30)
colors_s = plt.cm.RdBu_r(K_norm_s)
ax2.plot_surface(Us, Vs, Z_saddle, facecolors=colors_s, alpha=0.9, edgecolor='none')
ax2.set_title('Saddle: Gaussian Curvature $K$\n($K < 0$ everywhere)')
ax2.set_xlabel('x'); ax2.set_ylabel('y'); ax2.set_zlabel('z')

plt.suptitle('Gaussian Curvature Colormaps', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Torus K range: [{K_torus.min():.4f}, {K_torus.max():.4f}] (outer K>0, inner K<0)")
print(f"Saddle K range: [{K_saddle.min():.4f}, {K_saddle.max():.4f}] (always K<0)")

---
## 7. Geodesics on Surfaces

A **geodesic** is the surface analogue of a straight line — a curve that is locally the shortest path between its points. Geodesics satisfy the **geodesic equation**:

$$\ddot{u}^i + \Gamma^i_{jk} \dot{u}^j \dot{u}^k = 0, \qquad i = 1, 2$$

where $u^1 = u$, $u^2 = v$, and $\Gamma^i_{jk}$ are the **Christoffel symbols** of the second kind.

### Derivation from the Calculus of Variations

We seek curves $\gamma(t) = (u(t), v(t))$ that minimize the arc length:

$$L[\gamma] = \int_a^b \sqrt{E\dot{u}^2 + 2F\dot{u}\dot{v} + G\dot{v}^2} \, dt$$

Applying the Euler-Lagrange equations to this functional yields the geodesic equation. The Christoffel symbols are computed from the metric:

$$\Gamma^i_{jk} = \frac{1}{2} g^{il} \left( \frac{\partial g_{lj}}{\partial u^k} + \frac{\partial g_{lk}}{\partial u^j} - \frac{\partial g_{jk}}{\partial u^l} \right)$$

where $g_{ij} = \begin{bmatrix} E & F \\ F & G \end{bmatrix}$ and $g^{ij}$ is its inverse.

### Geodesic Shooting

To find geodesics, we convert the second-order ODE to a first-order system and integrate from an initial point $(u_0, v_0)$ with initial direction $(\dot{u}_0, \dot{v}_0)$. This is called **geodesic shooting**.

In [ ]:
def christoffel_symbols(sigma_func, u, v, eps=FINITE_DIFF_EPS):
    """Compute Christoffel symbols of the second kind at a point.

    Returns Gamma[i][j][k] = Gamma^i_{jk} for i,j,k in {0,1} (u=0, v=1).

    Args:
        sigma_func: Surface parametrization (u, v) -> R^3.
        u: First parameter. Scalar.
        v: Second parameter. Scalar.
        eps: Finite difference step. Scalar.

    Returns:
        Gamma: Christoffel symbols. Shape: (2, 2, 2).
    """
    # Compute metric at nearby points for derivatives of E, F, G
    def metric_at(u0, v0):
        E, F, G = first_fundamental_form(sigma_func, u0, v0, eps)
        return np.array([[E, F], [F, G]])

    g = metric_at(u, v)
    g_inv = np.linalg.inv(g)

    # Derivatives of metric tensor: dg[a][i][j] = d g_ij / d u^a
    dg = np.zeros((2, 2, 2))
    # d/du
    g_pu = metric_at(u + eps, v)
    g_mu = metric_at(u - eps, v)
    dg[0] = (g_pu - g_mu) / (2 * eps)
    # d/dv
    g_pv = metric_at(u, v + eps)
    g_mv = metric_at(u, v - eps)
    dg[1] = (g_pv - g_mv) / (2 * eps)

    # Christoffel symbols: Gamma^i_{jk} = 0.5 * g^{il} (dg_{lj}/du^k + dg_{lk}/du^j - dg_{jk}/du^l)
    Gamma = np.zeros((2, 2, 2))
    for i in range(2):
        for j in range(2):
            for k in range(2):
                val = 0.0
                for l in range(2):
                    val += g_inv[i, l] * (dg[k][l, j] + dg[j][l, k] - dg[l][j, k])
                Gamma[i, j, k] = 0.5 * val

    return Gamma


def shoot_geodesic(sigma_func, u0, v0, du0, dv0, s_max=GEODESIC_MAX_S, n_points=N_GEODESIC_POINTS):
    """Integrate a geodesic from initial point and direction.

    Solves the geodesic ODE: u'' + Gamma^i_{jk} u'^j u'^k = 0
    as a first-order system: state = [u, v, du/ds, dv/ds].

    Args:
        sigma_func: Surface parametrization (u, v) -> R^3.
        u0, v0: Initial parameter values. Scalars.
        du0, dv0: Initial parameter velocities. Scalars.
        s_max: Maximum arc length to integrate. Scalar.
        n_points: Number of output points. Integer.

    Returns:
        s_vals: Arc-length parameter values. Shape: (n_points,).
        uv_traj: Parameter trajectory. Shape: (n_points, 2).
        xyz_traj: Embedded trajectory in R^3. Shape: (n_points, 3).
    """
    def geodesic_ode(s, state):
        u, v, du, dv = state
        try:
            Gamma = christoffel_symbols(sigma_func, u, v)
        except Exception:
            return [du, dv, 0.0, 0.0]
        vel = np.array([du, dv])
        accel = np.zeros(2)
        for i in range(2):
            for j in range(2):
                for k in range(2):
                    accel[i] -= Gamma[i, j, k] * vel[j] * vel[k]
        return [du, dv, accel[0], accel[1]]

    s_span = (0, s_max)
    s_eval = np.linspace(0, s_max, n_points)
    y0 = [u0, v0, du0, dv0]

    sol = solve_ivp(geodesic_ode, s_span, y0, t_eval=s_eval,
                    method='RK45', max_step=s_max / 200, rtol=1e-8, atol=1e-10)

    uv_traj = sol.y[:2].T   # (n_points, 2)
    xyz_traj = np.array([sigma_func(uv[0], uv[1]) for uv in uv_traj])

    return sol.t, uv_traj, xyz_traj

In [ ]:
# ---- Geodesics on a sphere (should be great circles) ----
sphere_unit = lambda u, v: sphere(u, v, r=1.0)

fig = plt.figure(figsize=(16, 6))

# Left: sphere geodesics
ax1 = fig.add_subplot(1, 2, 1, projection='3d')

# Plot sphere wireframe
u_w = np.linspace(0, np.pi, 30)
v_w = np.linspace(0, 2 * np.pi, 30)
Uw, Vw = np.meshgrid(u_w, v_w)
ax1.plot_surface(np.sin(Uw)*np.cos(Vw), np.sin(Uw)*np.sin(Vw), np.cos(Uw),
                 alpha=0.15, color='lightblue', edgecolor='none')

# Shoot geodesics from near north pole in different directions
colors_geo = [C_BLUE, C_RED, C_GREEN, C_GOLD, C_PURPLE]
u0_sphere = 0.1  # near north pole
for idx, angle in enumerate(np.linspace(0, np.pi, 5, endpoint=False)):
    du0 = np.cos(angle)
    dv0 = np.sin(angle) / (np.sin(u0_sphere) + 1e-10)  # account for metric
    # Normalize to unit speed
    E, F, G = first_fundamental_form(sphere_unit, u0_sphere, 0.0)
    speed = np.sqrt(E * du0**2 + 2 * F * du0 * dv0 + G * dv0**2)
    du0 /= speed
    dv0 /= speed

    s_vals, uv, xyz = shoot_geodesic(sphere_unit, u0_sphere, 0.0, du0, dv0, s_max=np.pi)
    ax1.plot(xyz[:, 0], xyz[:, 1], xyz[:, 2], color=colors_geo[idx], linewidth=2)

ax1.set_title('Geodesics on Sphere\n(Great Circles)')
ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.set_zlabel('z')

# Right: geodesics on torus
ax2 = fig.add_subplot(1, 2, 2, projection='3d')

ax2.plot_surface(X_t, Y_t, Z_t, alpha=0.15, color='lightyellow', edgecolor='none')

# Shoot geodesics on torus
for idx, (du0, dv0) in enumerate([(1.0, 0.0), (0.0, 1.0), (1.0, 0.5), (1.0, 1.0), (0.5, 1.5)]):
    u0_t, v0_t = 0.0, 0.0
    E, F, G = first_fundamental_form(torus, u0_t, v0_t)
    speed = np.sqrt(E * du0**2 + 2 * F * du0 * dv0 + G * dv0**2)
    du0 /= speed
    dv0 /= speed
    s_vals, uv, xyz = shoot_geodesic(torus, u0_t, v0_t, du0, dv0, s_max=15.0, n_points=2000)
    ax2.plot(xyz[:, 0], xyz[:, 1], xyz[:, 2], color=colors_geo[idx], linewidth=1.5)

ax2.set_title('Geodesics on Torus')
ax2.set_xlabel('x'); ax2.set_ylabel('y'); ax2.set_zlabel('z')

plt.suptitle('Geodesic Shooting', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Geodesics on a Gaussian bump surface ----

def gaussian_bump(u, v, h=1.5, sigma=0.8):
    """Surface z = h * exp(-(u^2 + v^2)/(2*sigma^2))."""
    z = h * np.exp(-(u**2 + v**2) / (2 * sigma**2))
    return np.array([u, v, z])

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot surface
N_bump = 60
u_b = np.linspace(-2.5, 2.5, N_bump)
v_b = np.linspace(-2.5, 2.5, N_bump)
Ub, Vb = np.meshgrid(u_b, v_b)
Zb = 1.5 * np.exp(-(Ub**2 + Vb**2) / (2 * 0.8**2))
ax.plot_surface(Ub, Vb, Zb, alpha=0.3, cmap='terrain', edgecolor='none')

# Shoot geodesics across the bump
starts = [(-2.0, -1.0), (-2.0, 0.0), (-2.0, 1.0), (-1.5, -2.0)]
for idx, (u0, v0) in enumerate(starts):
    du0, dv0 = 1.0, 0.2 * (idx - 1.5)
    E, F, G = first_fundamental_form(gaussian_bump, u0, v0)
    speed = np.sqrt(E * du0**2 + 2 * F * du0 * dv0 + G * dv0**2)
    du0 /= speed; dv0 /= speed
    s_vals, uv, xyz = shoot_geodesic(gaussian_bump, u0, v0, du0, dv0, s_max=5.0)
    ax.plot(xyz[:, 0], xyz[:, 1], xyz[:, 2], color=colors_geo[idx], linewidth=2.5)
    ax.plot([xyz[0, 0]], [xyz[0, 1]], [xyz[0, 2]], 'o', color=colors_geo[idx], markersize=6)

ax.set_title('Geodesics on a Gaussian Bump Surface', fontsize=14, fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
plt.tight_layout()
plt.show()

print("Geodesics bend around the bump — the curved geometry of the surface")
print("deflects paths just like gravity bends light in general relativity.")

---
## 8. Application: Trajectory Smoothing on Surfaces

In robotics, we often need to plan paths on curved surfaces — a mobile robot traversing terrain, a tool moving along a workpiece, or a satellite orbit on a sphere. Given a set of **waypoints** on a surface, we want a **smooth path** that:

1. Passes near the waypoints (fidelity)
2. Has low curvature (smoothness / energy efficiency)

### Formulation

We parametrize the path as a spline in the $(u, v)$ parameter space, with control points $\{(u_i, v_i)\}$. The cost function is:

$$J = \underbrace{\sum_i \|path(s_i) - waypoint_i\|^2}_{\text{waypoint fidelity}} + \lambda \underbrace{\int \|\ddot{\gamma}(s)\|^2 ds}_{\text{smoothness}}$$

We minimize $J$ over the control points using numerical optimization. The path is then mapped to $\mathbb{R}^3$ via the surface parametrization $\sigma(u, v)$.

In [ ]:
def smooth_path_on_surface(sigma_func, waypoints_uv, n_control=15, n_eval=200,
                           lam_smooth=0.1, lam_fidelity=1.0):
    """Find a smooth path on a surface passing near given waypoints.

    Optimizes control points of a cubic spline in (u,v) parameter space.

    Args:
        sigma_func: Surface parametrization (u, v) -> R^3.
        waypoints_uv: Waypoint locations in parameter space. Shape: (n_wp, 2).
        n_control: Number of spline control points. Integer.
        n_eval: Number of evaluation points on the path. Integer.
        lam_smooth: Smoothness weight. Scalar.
        lam_fidelity: Fidelity weight. Scalar.

    Returns:
        path_uv: Optimized path in parameter space. Shape: (n_eval, 2).
        path_xyz: Optimized path in R^3. Shape: (n_eval, 3).
        raw_path_xyz: Raw piecewise-linear path through waypoints. Shape: (n_wp, 3).
    """
    n_wp = len(waypoints_uv)

    # Raw path (piecewise linear through waypoints)
    raw_path_xyz = np.array([sigma_func(wp[0], wp[1]) for wp in waypoints_uv])

    # Initial control points: linearly interpolate between first and last waypoint
    # with perturbation toward intermediate waypoints
    t_wp = np.linspace(0, 1, n_wp)
    t_ctrl = np.linspace(0, 1, n_control)

    # Initialize from linear interpolation of waypoints
    init_u = np.interp(t_ctrl, t_wp, waypoints_uv[:, 0])
    init_v = np.interp(t_ctrl, t_wp, waypoints_uv[:, 1])
    z0 = np.concatenate([init_u, init_v])

    def cost(z):
        u_ctrl = z[:n_control]
        v_ctrl = z[n_control:]

        # Build spline
        cs_u = CubicSpline(t_ctrl, u_ctrl)
        cs_v = CubicSpline(t_ctrl, v_ctrl)

        # Fidelity: distance to waypoints in R^3
        fidelity = 0.0
        for i, (t_w, wp) in enumerate(zip(t_wp, waypoints_uv)):
            u_at = cs_u(t_w)
            v_at = cs_v(t_w)
            pt_on_path = sigma_func(u_at, v_at)
            pt_target = sigma_func(wp[0], wp[1])
            fidelity += np.sum((pt_on_path - pt_target)**2)

        # Smoothness: integral of squared second derivative
        t_eval = np.linspace(0, 1, n_eval)
        u2 = cs_u(t_eval, 2)  # second derivative
        v2 = cs_v(t_eval, 2)
        smoothness = np.mean(u2**2 + v2**2)

        return lam_fidelity * fidelity + lam_smooth * smoothness

    result = minimize(cost, z0, method='L-BFGS-B', options={'maxiter': 500})

    u_opt = result.x[:n_control]
    v_opt = result.x[n_control:]
    cs_u = CubicSpline(t_ctrl, u_opt)
    cs_v = CubicSpline(t_ctrl, v_opt)

    t_eval = np.linspace(0, 1, n_eval)
    path_uv = np.column_stack([cs_u(t_eval), cs_v(t_eval)])
    path_xyz = np.array([sigma_func(uv[0], uv[1]) for uv in path_uv])

    return path_uv, path_xyz, raw_path_xyz


# ---- Trajectory smoothing on a terrain-like surface ----

def terrain(u, v):
    """Terrain-like surface: z = 0.4*sin(u)*cos(v) + 0.2*sin(2u)*sin(2v)."""
    z = 0.4 * np.sin(u) * np.cos(v) + 0.2 * np.sin(2 * u) * np.sin(2 * v)
    return np.array([u, v, z])

# Waypoints in parameter space
waypoints = np.array([
    [0.5, 0.5],
    [1.5, 1.0],
    [2.5, 2.5],
    [3.5, 1.5],
    [4.5, 3.0],
    [5.5, 2.0],
])

path_uv, path_xyz, raw_xyz = smooth_path_on_surface(
    terrain, waypoints, n_control=20, lam_smooth=0.05, lam_fidelity=5.0
)

print(f"Optimization complete. Path has {len(path_xyz)} points.")
print(f"Raw path length:    {np.sum(np.linalg.norm(np.diff(raw_xyz, axis=0), axis=1)):.3f}")
print(f"Smooth path length: {np.sum(np.linalg.norm(np.diff(path_xyz, axis=0), axis=1)):.3f}")

In [ ]:
# ---- Visualization: 3D surface + paths, and parameter space ----
fig = plt.figure(figsize=(16, 7))

# Left: 3D view
ax1 = fig.add_subplot(1, 2, 1, projection='3d')

# Plot terrain surface
N_ter = 60
u_ter = np.linspace(0, 6, N_ter)
v_ter = np.linspace(0, 4, N_ter)
Ut, Vt = np.meshgrid(u_ter, v_ter)
Zt = 0.4 * np.sin(Ut) * np.cos(Vt) + 0.2 * np.sin(2 * Ut) * np.sin(2 * Vt)
ax1.plot_surface(Ut, Vt, Zt, alpha=0.25, cmap='terrain', edgecolor='none')

# Waypoints
wp_xyz = np.array([terrain(wp[0], wp[1]) for wp in waypoints])
ax1.scatter(wp_xyz[:, 0], wp_xyz[:, 1], wp_xyz[:, 2],
            c=C_RED, s=80, zorder=10, marker='D', label='Waypoints')

# Raw path
ax1.plot(raw_xyz[:, 0], raw_xyz[:, 1], raw_xyz[:, 2],
         '--', color=C_GOLD, linewidth=2, label='Raw (piecewise linear)')

# Smooth path
ax1.plot(path_xyz[:, 0], path_xyz[:, 1], path_xyz[:, 2],
         color=C_BLUE, linewidth=3, label='Smoothed')

ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.set_zlabel('z')
ax1.set_title('Trajectory Smoothing on Terrain')
ax1.legend()

# Right: parameter space
ax2 = fig.add_subplot(1, 2, 2)
ax2.scatter(waypoints[:, 0], waypoints[:, 1],
            c=C_RED, s=80, zorder=10, marker='D', label='Waypoints')
ax2.plot(waypoints[:, 0], waypoints[:, 1],
         '--', color=C_GOLD, linewidth=2, label='Raw')
ax2.plot(path_uv[:, 0], path_uv[:, 1],
         color=C_BLUE, linewidth=3, label='Smoothed')
ax2.set_xlabel('$u$')
ax2.set_ylabel('$v$')
ax2.set_title('Parameter Space $(u, v)$')
ax2.legend()
ax2.set_aspect('equal')

plt.suptitle('Application: Trajectory Smoothing on a Curved Surface', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. Summary and Extensions

### Summary

| Concept | Formula / Key Idea | Robotics Application |
|---------|-------------------|---------------------|
| **Arc length** | $s = \int \|\alpha'\| dt$ | Trajectory parametrization |
| **Curvature** $\kappa$ | $\kappa = \|\alpha' \times \alpha''\| / \|\alpha'\|^3$ | Path smoothness metric |
| **Frenet-Serret frame** | $T, N, B$ with skew-symmetric ODE | Moving frame for end-effector |
| **First fundamental form** | $E, F, G$ metric tensor | Arc length and area on surfaces |
| **Gaussian curvature** $K$ | $K = (LN - M^2)/(EG - F^2)$ | Surface classification |
| **Geodesics** | $\ddot{u}^i + \Gamma^i_{jk}\dot{u}^j\dot{u}^k = 0$ | Shortest paths on surfaces |

### Extensions

- **Gauss-Bonnet Theorem:** $\int_M K \, dA = 2\pi\chi(M)$ — total curvature is a topological invariant
- **Riemannian Geometry:** generalize to abstract manifolds without embedding; define metrics intrinsically
- **Parallel Transport:** moving vectors along curves without twisting — critical for attitude interpolation on SO(3)
- **Information Geometry:** the Fisher information metric makes the space of probability distributions a Riemannian manifold (see [Information Geometry notebook](../probability/information-geometry/information_geometry.ipynb))
- **Trajectory Optimization on Manifolds:** combine geodesic methods with optimal control for planning on non-Euclidean configuration spaces

In [ ]:
# ---- Summary Figure: 2x2 panel ----
fig = plt.figure(figsize=(16, 12))

# (a) Frenet frame on helix
ax1 = fig.add_subplot(2, 2, 1, projection='3d')
ax1.plot(pts_helix[:, 0], pts_helix[:, 1], pts_helix[:, 2], color='gray', alpha=0.5, linewidth=1)
step_s = len(idx_fs) // 8
for i in range(0, len(idx_fs), step_s):
    p = pts_helix[idx_fs[i]]
    ax1.quiver(*p, *T_h[i] * 0.3, color=C_BLUE, arrow_length_ratio=0.15, linewidth=1.5)
    ax1.quiver(*p, *N_h[i] * 0.3, color=C_RED, arrow_length_ratio=0.15, linewidth=1.5)
    ax1.quiver(*p, *B_h[i] * 0.3, color=C_GREEN, arrow_length_ratio=0.15, linewidth=1.5)
ax1.set_title('(a) Frenet-Serret Frame')

# (b) Gaussian curvature on torus
ax2 = fig.add_subplot(2, 2, 2, projection='3d')
ax2.plot_surface(X_t, Y_t, Z_t, facecolors=colors_t, alpha=0.9, edgecolor='none')
ax2.set_title('(b) Gaussian Curvature on Torus')

# (c) Geodesics on sphere
ax3 = fig.add_subplot(2, 2, 3, projection='3d')
ax3.plot_surface(np.sin(Uw)*np.cos(Vw), np.sin(Uw)*np.sin(Vw), np.cos(Uw),
                 alpha=0.15, color='lightblue', edgecolor='none')
for idx, angle in enumerate(np.linspace(0, np.pi, 5, endpoint=False)):
    du0 = np.cos(angle)
    dv0 = np.sin(angle) / (np.sin(0.1) + 1e-10)
    E, F, G = first_fundamental_form(sphere_unit, 0.1, 0.0)
    speed = np.sqrt(E * du0**2 + 2 * F * du0 * dv0 + G * dv0**2)
    du0 /= speed; dv0 /= speed
    _, _, xyz_g = shoot_geodesic(sphere_unit, 0.1, 0.0, du0, dv0, s_max=np.pi, n_points=500)
    ax3.plot(xyz_g[:, 0], xyz_g[:, 1], xyz_g[:, 2], color=colors_geo[idx], linewidth=2)
ax3.set_title('(c) Geodesics on Sphere')

# (d) Trajectory smoothing
ax4 = fig.add_subplot(2, 2, 4, projection='3d')
ax4.plot_surface(Ut, Vt, Zt, alpha=0.2, cmap='terrain', edgecolor='none')
ax4.scatter(wp_xyz[:, 0], wp_xyz[:, 1], wp_xyz[:, 2], c=C_RED, s=60, marker='D')
ax4.plot(raw_xyz[:, 0], raw_xyz[:, 1], raw_xyz[:, 2], '--', color=C_GOLD, linewidth=2)
ax4.plot(path_xyz[:, 0], path_xyz[:, 1], path_xyz[:, 2], color=C_BLUE, linewidth=3)
ax4.set_title('(d) Trajectory Smoothing on Surface')

plt.suptitle('Differential Geometry: Summary', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()